# Surrogate Factory — UCFatigue
## Chapter 2. Data Acquisition
Objectives:
- Load the Excel dataset.
- Filter to one SSE element (FEM node `2110017`).
- Clean `Type_segment` whitespace (ETL Transform step).
- Save raw dataset.

### 0. Workflow initialisation

In [1]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

2026-06-25 12:24:17 - SurrogateFactoryLogs - INFO - Initializing Workflow...


2026-06-25 12:24:17 - SurrogateFactoryLogs - INFO - Setting up Workflow folders and paths...


2026-06-25 12:24:17 - SurrogateFactoryLogs - INFO - Changing working directory to Data folder: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data


2026-06-25 12:24:17 - SurrogateFactoryLogs - INFO - Loading default metadata schema...


2026-06-25 12:24:17 - SurrogateFactoryLogs - INFO - Tracker requested: 'mlflow'


...

Setting tracking uri: file:///Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/mlruns
2026-06-25 12:24:18 - SurrogateFactoryLogs - INFO - Adding methods to Catalog from configuration.


2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Workflow initialization completed successfully.


2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Resuming workflow job: 'UCFATIGUE_1'


2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Workflow data loaded successfully.


2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Workflow metadata successfully loaded for resume.


### 2. Data Acquisition — Extract / Transform / Load

In [2]:
workflow.import_metadata(stage_name="SF_2_Data_Acquisition_Generation")

2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Importing metadata for stage: 'SF_2_Data_Acquisition_Generation'


2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - Updating metadata for stage 'Data_Acquisition_Generation' in memory.


#### 2.1 Extract

In [3]:
from data_acquisition.outputs_parser import batch_extract
raw_data = batch_extract(workflow)
print(raw_data.shape)
raw_data.head()

2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'batch_extract'


Filtered to SSE=2110017: 869 rows
(869, 29)


,Operator,SSE,Mission,Seg,Flight_segment,Type_segment,FLAP,Altitude,TAS,Time,...,Vertical gust,Turn,Frontal gust,n0,Giro,Xcg(%CMA),Op-Mission,q,gamma,Mission_type
0,OpS,2110017,E1,7,C-1500,FLT-1,10,750,137.0,1.0,...,0.5197,6.276,NaN,1.92,0.670,NaN,OpS-E1,2976.266912,4.865686,Training
1,OpS,2110017,E1,8,CRUISE,FLT-2,0,1500,198.0,9.1,...,0.7994,6.102,NaN,1.85,0.881,NaN,OpS-E1,6080.727602,0.000000,Training
2,OpS,2110017,E1,9,APPROACH,FLT-3,15,750,119.0,1.0,...,0.3896,4.727,3.235,1.96,0.824,NaN,OpS-E1,2245.560006,0.000000,Training
3,OpS,2110017,E1,10,F/APPRO.,FLT-3,23,25,108.0,1.0,...,0.3896,4.727,3.235,1.96,0.824,NaN,OpS-E1,1889.367452,0.000000,Training
4,OpS,2110017,F1,9,C-1500,FLT-1,10,750,147.0,1.0,...,0.6058,7.365,NaN,1.82,0.864,NaN,OpS-F1,3426.615787,4.865686,Ferry


#### 2.2 Transform (strip whitespace from Type_segment)

In [4]:
from data_acquisition.outputs_parser import batch_transform
transformed = batch_transform(workflow, raw_data)
print("Type_segment unique values:", transformed['Type_segment'].unique().tolist())

2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'batch_transform'


Type_segment unique values: ['FLT-1', 'FLT-2', 'FLT-3', 'FLT-4', 'FLT-7', 'FLT-9', 'FLT-10', 'FLT-8', 'FLT-11']


#### 2.3 Load

In [5]:
from data_acquisition.outputs_parser import batch_load
Dataset = batch_load(workflow, transformed)
Dataset.describe()

2026-06-25 12:24:19 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'batch_load'


,SSE,Seg,FLAP,Altitude,TAS,Time,Distance,Pressure,THRUST,Mass,...,1g,Vertical maneuver,Vertical gust,Turn,Frontal gust,n0,Giro,Xcg(%CMA),q,gamma
count,869.0,869.000000,869.000000,869.000000,869.000000,869.000000,869.000000,361.000000,465.000000,869.000000,...,869.000000,869.000000,869.000000,869.000000,129.000000,869.000000,869.000000,821.000000,869.000000,869.000000
mean,2110017.0,42.211738,2.218642,9294.634062,190.871043,5.671646,18.736686,3.170886,7.148645,18874.777906,...,53.622806,19.137703,0.649379,6.362113,3.237240,1.939056,1.995853,19.463934,4510.245290,-0.266692
std,0.0,28.767722,5.369574,7701.901926,48.867640,24.658221,90.687607,1.906679,6.800328,1324.688731,...,8.296235,2.454225,0.168054,0.641480,0.196348,0.077678,1.532889,4.206660,1845.316550,4.573436
min,2110017.0,7.000000,0.000000,18.000000,101.000000,0.100000,0.200000,0.130000,-3.250000,14125.000000,...,31.561000,12.031000,0.346600,4.122000,2.542000,1.730000,0.662000,14.600000,1652.386607,-39.450765
25%,2110017.0,18.000000,0.000000,2000.000000,144.000000,0.490000,1.600000,1.430000,-0.060000,17761.000000,...,47.364000,17.185000,0.509700,6.010000,3.182000,1.900000,1.299000,16.400000,2881.202593,-4.704237
50%,2110017.0,35.000000,0.000000,8000.000000,197.571208,0.930000,2.500000,3.440000,7.320000,18905.000000,...,51.778000,18.893000,0.575000,6.382000,3.236000,1.940000,1.587000,18.500000,3579.988216,0.000000
75%,2110017.0,60.000000,0.000000,15000.000000,230.000000,3.000000,9.600000,4.980000,11.970000,19991.000000,...,61.558000,21.394000,0.806000,6.830000,3.350000,1.980000,1.874000,20.580000,6375.463109,2.944179
max,2110017.0,122.000000,23.000000,26000.000000,302.000000,623.000000,2325.000000,5.580000,23.470000,21886.000000,...,73.308000,24.471000,1.092000,7.530000,3.722000,2.190000,7.599000,29.300000,9637.860375,18.219357


#### 2.4 Visualisation

In [6]:
%matplotlib inline
from data_acquisition.outputs_parser import data_visualization
data_visualization(workflow, Dataset)

2026-06-25 12:24:20 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'data_visualization'


Profiling not available.


### Save

In [7]:
outputFilename = workflow.config['job_name'] + '_raw_dataset.csv'
workflow.save_data(Dataset, outputFilename)
workflow.save_metadata()

2026-06-25 12:24:20 - SurrogateFactoryLogs - INFO - File '/Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/UCFATIGUE_1_raw_dataset.csv' already exists — skipping save (delete data/ folder to rerun from scratch)


2026-06-25 12:24:20 - SurrogateFactoryLogs - INFO - Successfully saved workflow metadata to: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/artifacts/metadata_UCFATIGUE_1.json
